# Notebook 06 — LSA-Driven Anomaly Search

This notebook actively searches for anomaly candidates using TF-IDF + LSA as the feature space:

1. Normalize raw texts into an anomaly-focused view.
2. Build character n-gram TF-IDF vectors (3–5-gram, `char_wb` analyser).
3. Apply LSA (Truncated SVD, 256 dimensions) to obtain dense semantic vectors.
4. Run Isolation Forest and rank documents by anomaly score.

Lower scores mean more anomalous behaviour.

Reader guide:

- This notebook uses the semantic (LSA) feature space rather than structural features.
- See Notebook 05 and Notebook 08 for structural-feature-based anomaly detection.
- All ranking outputs are kept in `data/results/`.

## Setup

Add the `src/` directory to the Python path so that project modules are importable without installing the package.

In [ ]:
import sys
from pathlib import Path

project_root_path = Path.cwd().parent
if str(project_root_path / "src") not in sys.path:
    sys.path.insert(0, str(project_root_path / "src"))

In [1]:
from pathlib import Path

from core.data_io import ArticleDataset
from anomaly_detection import TextAnomalyDetector
from exploration import attach_original_text_by_doc_id, build_anomaly_candidate_table, sample_top_anomaly_texts
from preprocessing import TextNormalizer, TextPreprocessor

## Load dataset

I load raw data in original row order so document IDs stay aligned with every exported table.

In [2]:
project_root_path = Path.cwd().parent
results_data_directory_path = project_root_path / "data" / "results"
results_data_directory_path.mkdir(parents=True, exist_ok=True)

articles_csv_path = project_root_path / "data" / "raw" / "articles.csv"
articles_data_frame = ArticleDataset(input_csv_path=articles_csv_path).load_articles()

document_ids = articles_data_frame["doc_id"].tolist()
raw_texts = articles_data_frame["text"].tolist()

articles_data_frame.head()

,doc_id,text
0,DOC_00001,"When I first saw this, I thought for a second ..."
1,DOC_00002,The difficulties of a high Isp OTV include: Lo...
2,DOC_00003,"Forwarded from Neal Ausman, Galileo Mission Di..."
3,DOC_00004,Sjogren's syndrome has been known to induce dr...
4,DOC_00005,"Yes, I want to concentrate on other developmen..."


## Build anomaly features with TF-IDF + LSA

Why this setup:

- Character n-grams capture unusual formatting, repetition, and symbol patterns.
- LSA reduces sparse noise and gives a dense space for robust anomaly scoring.

In [3]:
text_normalizer = TextNormalizer()
normalized_bundle = text_normalizer.normalize_for_both_tasks(raw_texts)

anomaly_preprocessor = TextPreprocessor(
    vectorization_model_name="tfidf_lsa_dense",
    max_features=30000,
    min_document_frequency=1,
    max_document_frequency=1.0,
    ngram_range=(3, 5),
    analyzer_mode="char_wb",
    dense_embedding_dimension=256,
    random_seed=42,
)

# Fit vectorizer + LSA on anomaly-oriented normalized text.
anomaly_feature_matrix = anomaly_preprocessor.fit_transform(normalized_bundle.anomaly_texts)
anomaly_feature_matrix.shape

(2164, 256)

## Score anomalies

I run Isolation Forest and create a ranked candidate table.

Interpretation:

- Top rows are strongest anomaly candidates.
- `predicted_anomaly` shows model binary prediction using configured contamination.

In [4]:
anomaly_detector = TextAnomalyDetector(contamination_ratio=0.02, random_seed=42)
anomaly_mask, anomaly_scores = anomaly_detector.run_detection(anomaly_feature_matrix)

anomaly_candidate_table = build_anomaly_candidate_table(
    document_ids=document_ids,
    anomaly_scores=anomaly_scores.tolist(),
    anomaly_mask=anomaly_mask.tolist(),
)
anomaly_candidate_table_with_text = attach_original_text_by_doc_id(
    anomaly_table=anomaly_candidate_table,
    document_ids=document_ids,
    raw_texts=raw_texts,
)
anomaly_candidate_table_with_text.to_csv(results_data_directory_path / "notebook_06_anomaly_candidates.csv", index=False)

anomaly_candidate_table_with_text.head(20)

,doc_id,score,predicted_anomaly,anomaly_rank,text
0,DOC_00443,-0.487344,True,1,Am I justified in being pissed off at this doc...
1,DOC_00909,-0.486837,True,2,Les Bartel's comments: Let me add my .02 in. I...
2,DOC_01833,-0.485884,True,3,: Am I justified in being pissed off at this d...
3,DOC_01603,-0.485483,True,4,Some recent postings remind me that I had read...
4,DOC_00223,-0.483222,True,5,Help!! I need code/package/whatever to take 3-...
5,DOC_00718,-0.482727,True,6,+At one time there was speculation that the fi...
6,DOC_00192,-0.482307,True,7,+++ ++Once inflated the substance was no longe...
7,DOC_01025,-0.480305,True,8,I need as much information about Cosmos 2238 a...
8,DOC_02026,-0.479968,True,9,Does anyone know ifthe STS-56 email press kit ...
9,DOC_00085,-0.479878,True,10,At one time there was speculation that the fir...


## Inspect top candidates

Use this section for qualitative report evidence.

I sample top-ranked documents with small snippets so the outlier pattern can be inspected manually.

In [5]:
top_anomaly_samples = sample_top_anomaly_texts(
    anomaly_candidate_table=anomaly_candidate_table,
    document_ids=document_ids,
    raw_texts=raw_texts,
    top_k=10,
)
top_anomaly_samples_with_text = attach_original_text_by_doc_id(
    anomaly_table=top_anomaly_samples,
    document_ids=document_ids,
    raw_texts=raw_texts,
)
top_anomaly_samples_with_text.to_csv(results_data_directory_path / "notebook_06_top_anomaly_samples.csv", index=False)

top_anomaly_samples_with_text[["anomaly_rank", "doc_id", "score", "predicted_anomaly", "snippet"]]

,anomaly_rank,doc_id,score,predicted_anomaly,snippet
0,1,DOC_00443,-0.487344,True,Am I justified in being pissed off at this doc...
1,2,DOC_00909,-0.486837,True,Les Bartel's comments: Let me add my .02 in. I...
2,3,DOC_01833,-0.485884,True,: Am I justified in being pissed off at this d...
3,4,DOC_01603,-0.485483,True,Some recent postings remind me that I had read...
4,5,DOC_00223,-0.483222,True,Help!! I need code/package/whatever to take 3-...
5,6,DOC_00718,-0.482727,True,+At one time there was speculation that the fi...
6,7,DOC_00192,-0.482307,True,+++ ++Once inflated the substance was no longe...
7,8,DOC_01025,-0.480305,True,I need as much information about Cosmos 2238 a...
8,9,DOC_02026,-0.479968,True,Does anyone know ifthe STS-56 email press kit ...
9,10,DOC_00085,-0.479878,True,At one time there was speculation that the fir...


## Optional: Create submission-style top-50 table

This helper block builds a deterministic top-50 candidate output in assignment format.

In [6]:
submission_top_50 = anomaly_candidate_table.head(50).copy()
submission_top_50["anomaly"] = submission_top_50.index + 1
submission_table = submission_top_50[["anomaly", "doc_id"]]
submission_table_with_text = attach_original_text_by_doc_id(
    anomaly_table=submission_table,
    document_ids=document_ids,
    raw_texts=raw_texts,
)
submission_table_with_text.to_csv(results_data_directory_path / "notebook_06_anomalies_top_50.csv", index=False)
submission_table_with_text.head()

,anomaly,doc_id,text
0,1,DOC_00443,Am I justified in being pissed off at this doc...
1,2,DOC_00909,Les Bartel's comments: Let me add my .02 in. I...
2,3,DOC_01833,: Am I justified in being pissed off at this d...
3,4,DOC_01603,Some recent postings remind me that I had read...
4,5,DOC_00223,Help!! I need code/package/whatever to take 3-...


### Files produced by this notebook

| File | Description |
|---|---|
| `data/results/notebook_06_anomaly_candidates.csv` | Full ranked anomaly table with raw text |
| `data/results/notebook_06_top_anomaly_samples.csv` | Top-10 anomaly documents with snippets |
| `data/results/notebook_06_anomalies_top_50.csv` | Submission-style top-50 anomaly list with raw text |